# Entregable 2 — Navegación en grilla 9×9 con Qwen2.5-1.5B-Instruct
**Grupo 2** ·

Antes de empezar: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Luego ejecutar las celdas en orden.

## 0. Guardar resultados en Google Drive
Los archivos de `/content` se borran si Colab reinicia el entorno. Todo se guarda en Drive para poder retomar la evaluación.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/GenIA_Entregable2"
except ImportError:          # fuera de Colab (ej. local)
    OUT_DIR = "resultados_entregable2"
os.makedirs(OUT_DIR, exist_ok=True)
print("Resultados en:", OUT_DIR)


Mounted at /content/drive
Resultados en: /content/drive/MyDrive/GenIA_Entregable2


## 1. Cargar el modelo (versión fijada)



In [ ]:
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi

assert torch.cuda.is_available(), "No hay GPU: activa la T4 en Entorno de ejecución"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"   # <- pegar aquí el hash impreso abajo, entre comillas

if MODEL_REVISION is None:
    MODEL_REVISION = HfApi().model_info(MODEL_NAME).sha
    print(f"Hash actual del modelo: {MODEL_REVISION}  <- cópienlo en MODEL_REVISION")

print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION,
                                             torch_dtype=torch.float16,  # la T4 no soporta bf16
                                             device_map="auto")
model.eval()
print(f"Modelo cargado: {MODEL_NAME} @ {MODEL_REVISION}")

def generar(prompt, max_new_tokens):
    """Una llamada al modelo con decodificación greedy (determinista)."""
    messages = [{"role": "user", "content": prompt}]
    texto = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


torch 2.11.0+cu128 | transformers 5.16.1 | GPU Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo cargado: Qwen/Qwen2.5-1.5B-Instruct @ 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


## 2. Laberintos
El laberinto 0 es **el mismo del Entregable 1** (inicio (0,0), meta (0,8)). Como el Entregable 1 tenía un solo laberinto, el set se completa con laberintos del mismo formato (9×9, 0 = libre, 1 = muro) generados con semilla fija: laberinto perfecto con 3 muros abiertos para crear caminos alternativos, e inicio/meta aleatorios a distancia mínima 10. *Esta ampliación se declara en el PDF.*

In [ ]:
import random, re, json, time
from collections import deque

# ---------- Laberintos ----------
LAB_D1 = [
    [0,0,1,0,0,0,0,0,0],
    [1,0,1,0,1,1,1,1,0],
    [0,0,0,0,1,0,0,1,0],
    [0,1,1,1,1,0,1,1,0],
    [0,0,0,0,0,0,0,1,0],
    [1,1,1,1,1,1,0,1,0],
    [0,0,0,0,0,1,0,0,0],
    [0,1,1,1,0,1,1,1,1],
    [0,0,0,0,0,0,0,0,0],
]

def bfs(lab, inicio, meta):
    n = len(lab)
    prev = {inicio: None}
    q = deque([inicio])
    while q:
        r, c = q.popleft()
        if (r, c) == meta:
            camino = []
            p = meta
            while p is not None:
                camino.append(p); p = prev[p]
            return camino[::-1]
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n and 0 <= nc < n and lab[nr][nc] == 0 and (nr,nc) not in prev:
                prev[(nr,nc)] = (r,c); q.append((nr,nc))
    return None

def generar_laberinto(seed, n=9, muros_extra=3, dist_min=10):
    """Laberinto perfecto (backtracker) en 9x9 + algunos muros abiertos para crear ciclos."""
    rng = random.Random(seed)
    lab = [[1]*n for _ in range(n)]
    k = (n+1)//2
    visit = {(0,0)}; pila = [(0,0)]; lab[0][0] = 0
    while pila:
        i, j = pila[-1]
        vec = [(i+di, j+dj) for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]
               if 0 <= i+di < k and 0 <= j+dj < k and (i+di, j+dj) not in visit]
        if not vec:
            pila.pop(); continue
        ni, nj = rng.choice(vec)
        lab[i+ni][j+nj] = 0          # celda muro entre ambas
        lab[2*ni][2*nj] = 0
        visit.add((ni, nj)); pila.append((ni, nj))
    # abrir algunos muros internos para que haya más de un camino
    candidatos = [(r,c) for r in range(n) for c in range(n)
                  if lab[r][c] == 1 and ((r%2==0 and c%2==1) or (r%2==1 and c%2==0))]
    for r, c in rng.sample(candidatos, min(muros_extra, len(candidatos))):
        lab[r][c] = 0
    libres = [(r,c) for r in range(n) for c in range(n) if lab[r][c] == 0]
    while True:
        a, b = rng.sample(libres, 2)
        camino = bfs(lab, a, b)
        if camino and len(camino)-1 >= dist_min:
            return lab, a, b

def construir_dataset(N, seed_base=2026):
    data = [{"id": 0, "laberinto": LAB_D1, "inicio": (0,0), "meta": (0,8), "origen": "Entregable 1"}]
    for i in range(1, N):
        lab, a, b = generar_laberinto(seed_base + i)
        data.append({"id": i, "laberinto": lab, "inicio": a, "meta": b, "origen": f"seed {seed_base+i}"})
    return data


## 3. Verificador
Mismo criterio de corrección del Entregable 1 para todos los métodos: fracasa si pisa un muro, salta a una celda no adyacente, sale del rango o no llega a la meta.

In [ ]:
# ---------- Verificador (mismo criterio para todos los métodos) ----------
def evaluar_ruta(lab, inicio, meta, ruta):
    n = len(lab)
    if not ruta:
        return "sin_ruta", []
    ruta = list(ruta)
    if ruta[0] != inicio:
        ruta = [inicio] + ruta
    for idx, (r, c) in enumerate(ruta):
        if not (0 <= r < n and 0 <= c < n):
            return "fuera_de_rango", ruta[:idx+1]
        if lab[r][c] == 1:
            return "pisa_muro", ruta[:idx+1]
        if idx > 0:
            pr, pc = ruta[idx-1]
            if abs(r-pr) + abs(c-pc) != 1:
                return "salto_no_adyacente", ruta[:idx+1]
        if (r, c) == meta:
            return "exito", ruta[:idx+1]
    return "no_llega_meta", ruta


## 4. Baseline: direct prompting (prompt del Entregable 1)
Se pide la ruta completa en una sola pasada. El parser toma la primera coordenada de cada línea e ignora las líneas de verificación de adyacencia (que mencionan la celda anterior).

In [ ]:
# ---------- Baseline: parser de la salida libre ----------
PATRON = re.compile(r"\(\s*(\d+)\s*,\s*(\d+)\s*\)")
PALABRAS_IGNORAR = ("adyac", "desde", "anterior", "previa")

def parsear_ruta_libre(texto):
    ruta = []
    for linea in texto.splitlines():
        if any(p in linea.lower() for p in PALABRAS_IGNORAR):
            continue
        m = PATRON.search(linea)
        if m:
            p = (int(m.group(1)), int(m.group(2)))
            if not ruta or ruta[-1] != p:
                ruta.append(p)
    return ruta

def prompt_baseline(lab, inicio, meta):
    filas, columnas = len(lab), len(lab[0])
    matriz_str = "\n".join(str(f) for f in lab)
    return f"""
Dada la siguiente matriz de un laberinto de TAMAÑO ESPECÍFICO {filas} filas por {columnas} columnas:

{matriz_str}

RESTRICCIONES DE DIMENSIÓN Y MAPA:
- El mapa tiene exactamente {filas} filas (índices 0 a {filas-1}) y {columnas} columnas (índices 0 a {columnas-1}).
- '0' es un camino libre y '1' es una muralla infranqueable.
- Inicio: ({inicio[0]}, {inicio[1]}) [Fila {inicio[0]}, Columna {inicio[1]}]
- Meta: ({meta[0]}, {meta[1]}) [Fila {meta[0]}, Columna {meta[1]}]

REGLA ESTRICTA DE DIAGNÓSTICO:
Propón una ruta para llegar a la meta. Para CADA PASO que escribas, debes incluir obligatoriamente:
1. La coordenada propuesta: (Fila, Columna).
2. El valor real en la matriz para esa coordenada (¿Es 0 o 1?).
3. Verificación de adyacencia: Confirma que el movimiento es exactamente a 1 casilla de distancia adyacente (Arriba, Abajo, Izquierda o Derecha) desde la coordenada anterior.

Escribe la solución paso a paso.
"""

def correr_baseline(generar, lab, inicio, meta, max_new_tokens=1000):
    t0 = time.time()
    salida = generar(prompt_baseline(lab, inicio, meta), max_new_tokens)
    ruta = parsear_ruta_libre(salida)
    estado, ruta_eval = evaluar_ruta(lab, inicio, meta, ruta)
    return {"estado": estado, "ruta": ruta_eval, "pasos": max(len(ruta_eval)-1, 0),
            "invalidas": None, "valor_mal_declarado": None, "coordenada_mal_declarada": None,
            "tiempo_s": round(time.time()-t0, 1), "salida_cruda": salida}


## 5. Solución: agente paso a paso
En cada paso el entorno le entrega al modelo su posición, la meta, los últimos pasos y los 4 vecinos con su valor real y si ya fueron visitados. Los 4 movimientos se presentan con su celda y su valor real (incluidos muros y bordes, para que evitarlos siga siendo decisión del modelo), junto con la dirección en que está la meta. El modelo responde en una línea: dirección de la meta → movimiento elegido (Arriba/Abajo/Izquierda/Derecha) → los tres campos de esa celda. Si un movimiento es rechazado, se marca como RECHAZADO en el siguiente intento. Se registra además si la coordenada o el valor declarados no coinciden con la celda elegida.

*Versiones descartadas (estrategias alternativas):* **v1** pedir la coordenada directamente → el modelo copiaba su posición actual y no avanzaba (3/3 laberintos). **v2** opciones con letra A–D → avanzaba, pero con sesgo hacia la opción C y bucles. **v3** v2 + dirección de la meta → el sesgo hacia C se mantuvo. **v4** v3 + razonamiento corto → el modelo identificaba bien la dirección de la meta (3/3), pero no la traducía a la letra correcta y listaba muros como libres. **v5 (actual)**: el modelo elige el movimiento por su nombre, con las mismas palabras que la dirección de la meta.

Con `usar_memoria=False` se obtiene la **estrategia alternativa**: el mismo agente, pero sin historial ni marcas de celdas visitadas. Sirve para medir cuánto aporta la memoria contra los bucles.

In [ ]:
# ---------- Solución: agente paso a paso ----------
DIRS = [("Arriba", -1, 0), ("Abajo", 1, 0), ("Izquierda", 0, -1), ("Derecha", 0, 1)]
LETRAS = "ABCD"
PATRON_VALOR = re.compile(r"(?:valor\s*:?\s*|\)\s*\|\s*)(\d)\b", re.IGNORECASE)
PATRON_LETRA = re.compile(r"\W*(?:[Oo]pci[oó]n\s*)?([ABCD])\b")

def opciones(lab, pos):
    """Las 4 opciones de movimiento: (letra, nombre, celda o None si está fuera del mapa)."""
    n = len(lab); ops = []
    for letra, (nombre, dr, dc) in zip(LETRAS, DIRS):
        r, c = pos[0]+dr, pos[1]+dc
        ops.append((letra, nombre, (r, c) if 0 <= r < n and 0 <= c < n else None))
    return ops

def direccion_meta(pos, meta):
    partes = []
    df, dc = meta[0]-pos[0], meta[1]-pos[1]
    if df: partes.append(f"{'Abajo' if df > 0 else 'Arriba'} ({abs(df)} filas)")
    if dc: partes.append(f"{'Derecha' if dc > 0 else 'Izquierda'} ({abs(dc)} columnas)")
    return " y ".join(partes)

NOMBRE_A_LETRA = {"arriba": "A", "abajo": "B", "izquierda": "C", "derecha": "D"}
LETRA_A_NOMBRE = {"A": "Arriba", "B": "Abajo", "C": "Izquierda", "D": "Derecha"}
PATRON_NOMBRE = re.compile(r"\W*(arriba|abajo|izquierda|derecha)\b", re.IGNORECASE)

def prompt_paso(lab, pos, meta, visitadas, historial, error=None, usar_memoria=True, rechazadas=(),
                usar_direccion=True):
    n = len(lab)
    matriz_str = "\n".join(str(f) for f in lab)
    lineas = []
    for letra, nombre, celda in opciones(lab, pos):
        if celda is None:
            linea = f"- {nombre}: fuera del mapa"
        else:
            v = lab[celda[0]][celda[1]]
            linea = f"- {nombre}: ({celda[0]}, {celda[1]}) -> valor {v} [{'muro' if v == 1 else 'libre'}]"
            if usar_memoria and v == 0 and celda in visitadas:
                linea += " (ya visitada)"
        if letra in rechazadas:
            linea += "  <- RECHAZADA"
        lineas.append(linea)
    texto = f"""Estás navegando un laberinto de {n}x{n} (0 = libre, 1 = muro infranqueable).

Mapa:
{matriz_str}

Estás en ({pos[0]}, {pos[1]}). La meta es ({meta[0]}, {meta[1]}).
"""
    if usar_direccion:
        texto += f"La meta está hacia: {direccion_meta(pos, meta)}.\n"
    if usar_memoria:
        texto += "Últimos pasos: " + " -> ".join(f"({r}, {c})" for r, c in historial[-8:]) + "\n"
    texto += "\nMovimientos posibles:\n" + "\n".join(lineas) + "\n\n"
    texto += "Elige un movimiento hacia una celda libre que te acerque a la meta. No elijas muros ni movimientos fuera del mapa.\n"
    if usar_memoria:
        texto += "Prefiere celdas no visitadas. Si todas las celdas libres ya fueron visitadas, estás en un callejón sin salida: retrocede a una celda ya visitada.\n"
    if error:
        texto += f"\nTu propuesta anterior fue RECHAZADA: {error}. Elige otro movimiento.\n"
    texto += """
Responde en UNA sola línea con este formato:
Meta hacia: DIRECCIÓN | Elijo: MOVIMIENTO | (fila, columna) | valor: V | adyacente: sí
donde MOVIMIENTO es Arriba, Abajo, Izquierda o Derecha."""
    return texto

def validar_propuesta(lab, pos, texto):
    """Devuelve (celda, error, valor_declarado, coordenada_mal_declarada, letra).
    La decisión es el movimiento escrito después de "Elijo" (palabra o letra); si no hay,
    se usa la primera coordenada distinta de la posición actual."""
    n = len(lab)
    partes = re.split(r"[Ee]lijo", texto)
    if len(partes) > 1:
        texto = partes[-1]
    ops = {l: celda for l, _, celda in opciones(lab, pos)}
    coords = [(int(a), int(b)) for a, b in PATRON.findall(texto)]
    mv = PATRON_VALOR.search(texto)
    valor_decl = int(mv.group(1)) if mv else None
    mn = PATRON_NOMBRE.match(texto)
    ml = PATRON_LETRA.match(texto)
    letra = NOMBRE_A_LETRA[mn.group(1).lower()] if mn else (ml.group(1) if ml else None)
    if letra:
        celda = ops[letra]
        if celda is None:
            return None, f"{LETRA_A_NOMBRE[letra]} está fuera del mapa", valor_decl, False, letra
        coord_mal = bool(coords) and coords[0] != celda
    else:
        candidatas = [p for p in coords if p != pos]
        if not candidatas:
            return None, "no indicaste un movimiento (Arriba, Abajo, Izquierda o Derecha)", valor_decl, False, None
        celda = candidatas[0]; coord_mal = False
        if not (0 <= celda[0] < n and 0 <= celda[1] < n):
            return None, f"({celda[0]}, {celda[1]}) está fuera del mapa", valor_decl, False, None
        if abs(celda[0]-pos[0]) + abs(celda[1]-pos[1]) != 1:
            return None, f"({celda[0]}, {celda[1]}) no es adyacente a ({pos[0]}, {pos[1]})", valor_decl, False, None
        letra = next((l for l, c in ops.items() if c == celda), None)
    if lab[celda[0]][celda[1]] == 1:
        return None, f"{LETRA_A_NOMBRE.get(letra, '')} ({celda[0]}, {celda[1]}) es un muro (valor 1)".strip(), valor_decl, coord_mal, letra
    return celda, None, valor_decl, coord_mal, letra

def correr_agente(generar, lab, inicio, meta, max_pasos=81, max_reintentos=3,
                  max_new_tokens=60, verbose=False, usar_memoria=True, usar_direccion=True):
    t0 = time.time()
    pos = inicio; ruta = [inicio]; visitadas = {inicio}
    invalidas = 0; valor_mal = 0; coord_mal_total = 0; log = []; estado = None
    while pos != meta and len(ruta)-1 < max_pasos:
        error = None; nuevo = None; rechazadas = set()
        for intento in range(max_reintentos + 1):
            salida = generar(prompt_paso(lab, pos, meta, visitadas, ruta, error, usar_memoria, rechazadas, usar_direccion),
                             max_new_tokens)
            nuevo, error, valor_decl, coord_mal, letra = validar_propuesta(lab, pos, salida)
            log.append({"pos": pos, "intento": intento, "salida": salida.strip(), "error": error})
            if verbose:
                marca = "OK " if error is None else "RECHAZADO"
                print(f"  {marca} en {pos}: {salida.strip()[:140]}" + (f"  -> {error}" if error else ""))
            coord_mal_total += int(coord_mal)
            if error is None:
                if valor_decl is not None and valor_decl != lab[nuevo[0]][nuevo[1]]:
                    valor_mal += 1
                break
            invalidas += 1
            if letra:
                rechazadas.add(letra)
        if nuevo is None:
            estado = "propuestas_invalidas"
            break
        pos = nuevo; ruta.append(pos); visitadas.add(pos)
    if estado is None:
        estado, ruta = evaluar_ruta(lab, inicio, meta, ruta)
        if estado == "no_llega_meta":
            estado = "limite_pasos"
    return {"estado": estado, "ruta": ruta, "pasos": len(ruta)-1, "invalidas": invalidas,
            "valor_mal_declarado": valor_mal, "coordenada_mal_declarada": coord_mal_total,
            "tiempo_s": round(time.time()-t0, 1), "salida_cruda": log}


## 6. Control aleatorio y herramientas de análisis
El control mide cuánto logra el entorno *sin* el modelo: si el agente lo supera, la mejora es atribuible a las decisiones del modelo.

In [ ]:
# ---------- Control: política aleatoria en el mismo entorno ----------
def correr_aleatorio(lab, inicio, meta, max_pasos=81, seed=0):
    rng = random.Random(seed); n = len(lab)
    pos = inicio; ruta = [inicio]
    while pos != meta and len(ruta)-1 < max_pasos:
        opciones = [(pos[0]+dr, pos[1]+dc) for _, dr, dc in DIRS
                    if 0 <= pos[0]+dr < n and 0 <= pos[1]+dc < n and lab[pos[0]+dr][pos[1]+dc] == 0]
        pos = rng.choice(opciones); ruta.append(pos)
    estado, ruta = evaluar_ruta(lab, inicio, meta, ruta)
    if estado == "no_llega_meta":
        estado = "limite_pasos"
    return {"estado": estado, "ruta": ruta, "pasos": len(ruta)-1, "invalidas": 0,
            "valor_mal_declarado": 0, "coordenada_mal_declarada": 0, "tiempo_s": 0.0, "salida_cruda": ""}

def dibujar(lab, ruta, inicio, meta):
    en_ruta = set(ruta)
    for r, fila in enumerate(lab):
        s = ""
        for c, v in enumerate(fila):
            if (r, c) == inicio: s += " I"
            elif (r, c) == meta: s += " M"
            elif v == 1: s += " █"
            elif (r, c) in en_ruta: s += " ·"
            else: s += "  "
        print(s)

def analizar_fallo(lab, ruta):
    """Datos para explicar POR QUÉ falla: revisitas y callejones sin salida."""
    from collections import Counter
    n = len(lab)
    cuenta = Counter(ruta)
    revisitadas = [(p, k) for p, k in cuenta.most_common() if k > 1]
    def libres(p):
        return sum(1 for _, dr, dc in DIRS
                   if 0 <= p[0]+dr < n and 0 <= p[1]+dc < n and lab[p[0]+dr][p[1]+dc] == 0)
    callejones = sorted({p for p in ruta if libres(p) == 1})
    print(f"Pasos totales: {len(ruta)-1} | celdas distintas: {len(cuenta)}")
    print("Celdas más revisitadas (celda, veces):", revisitadas[:6])
    print("Callejones sin salida que visitó:", callejones)


## 7. Prueba rápida (3 laberintos)
Corre esto primero. Confirma que todo funciona, revisa que la *ruta leída* del baseline coincida con lo que escribió el modelo, y mira los tiempos para decidir N.

In [ ]:
datos_prueba = construir_dataset(3)
for x in datos_prueba:
    lab, a, b = x["laberinto"], tuple(x["inicio"]), tuple(x["meta"])
    print(f"\n===== Laberinto {x['id']} ({x['origen']}) | inicio {a} meta {b} | óptimo {len(bfs(lab,a,b))-1} pasos")
    rb = correr_baseline(generar, lab, a, b)
    print("BASELINE:", rb["estado"], "| tiempo:", rb["tiempo_s"], "s | ruta leída:", rb["ruta"][:12])
    print("--- salida cruda baseline ---\n", rb["salida_cruda"][:800], "\n---")
    ra = correr_agente(generar, lab, a, b)
    print("AGENTE:  ", ra["estado"], "| pasos:", ra["pasos"], "| inválidas:", ra["invalidas"], "| tiempo:", ra["tiempo_s"], "s")
    print("--- primeras respuestas del agente ---")
    for e in ra["salida_cruda"][:6]:
        print(f"  en {e['pos']} intento {e['intento']}: {e['salida'][:140]}  -> {e['error'] or 'OK'}")



===== Laberinto 0 (Entregable 1) | inicio (0, 0) meta (0, 8) | óptimo 12 pasos
BASELINE: pisa_muro | tiempo: 27.3 s | ruta leída: [(0, 0), (0, 1), (1, 1), (1, 2)]
--- salida cruda baseline ---
 ### Paso 1: Coordenadas Inicial y Valor Matriz
**Coordenada:** (0, 0)
**Valor en Matriz:** 0

### Paso 2: Movimiento hacia arriba
**Coordenada:** (0, 1)
**Valor en Matriz:** 0

### Paso 3: Movimiento hacia derecha
**Coordenada:** (1, 1)
**Valor en Matriz:** 0

### Paso 4: Movimiento hacia abajo
**Coordenada:** (1, 2)
**Valor en Matriz:** 0

### Paso 5: Movimiento hacia izquierda
**Coordenada:** (0, 2)
**Valor en Matriz:** 0

### Paso 6: Movimiento hacia arriba
**Coordenada:** (0, 3)
**Valor en Matriz:** 0

### Paso 7: Movimiento hacia derecha
**Coordenada:** (1, 3)
**Valor en Matriz:** 0

### Paso 8: Movimiento hacia abajo
**Coordenada:** (1, 4)
**Valor en Matriz:** 0

### Paso 9: Movimiento hacia izquierda
**Coordenada:** (0, 4)
**Valor en Matriz:** 0

### Paso 10: Movimiento hacia arriba 
---

## 8. Evaluación completa
Corre 4 métodos sobre los mismos N laberintos y guarda cada resultado en Drive apenas termina. Si Colab se desconecta: volver a ejecutar las celdas 0 a 6 y luego esta; retoma donde quedó.

- `exito`: criterio del Entregable 1 aplicado a la ruta final.
- `exito_estricto`: además, **cero propuestas inválidas** (el modelo nunca propuso un muro, un salto ni una celda fuera de rango). Es el criterio sin la ayuda de los reintentos.

In [ ]:
import pandas as pd

N = 20
METODOS = ["baseline", "agente", "agente_sin_memoria", "aleatorio"]
CSV = os.path.join(OUT_DIR, "resultados.csv")
JSONL = os.path.join(OUT_DIR, "salidas_crudas.jsonl")

datos = construir_dataset(N)
with open(os.path.join(OUT_DIR, "laberintos_9x9.json"), "w") as f:
    json.dump(datos, f)

hechos = set()
if os.path.exists(CSV):
    prev = pd.read_csv(CSV)
    hechos = set(zip(prev["id"], prev["metodo"]))

for x in datos:
    lab, a, b = x["laberinto"], tuple(x["inicio"]), tuple(x["meta"])
    optimo = len(bfs(lab, a, b)) - 1
    for metodo in METODOS:
        if (x["id"], metodo) in hechos:
            continue
        if metodo == "baseline":
            r = correr_baseline(generar, lab, a, b)
        elif metodo == "agente":
            r = correr_agente(generar, lab, a, b, usar_memoria=True)
        elif metodo == "agente_sin_memoria":
            r = correr_agente(generar, lab, a, b, usar_memoria=False)
        else:
            r = correr_aleatorio(lab, a, b, seed=x["id"])
        exito = r["estado"] == "exito"
        fila = {"id": x["id"], "metodo": metodo, "estado": r["estado"], "exito": exito,
                "exito_estricto": exito and not r["invalidas"],
                "pasos": r["pasos"], "optimo": optimo, "invalidas": r["invalidas"],
                "valor_mal_declarado": r["valor_mal_declarado"],
                "coordenada_mal_declarada": r["coordenada_mal_declarada"], "tiempo_s": r["tiempo_s"]}
        pd.DataFrame([fila]).to_csv(CSV, mode="a", header=not os.path.exists(CSV), index=False)
        with open(JSONL, "a") as f:
            f.write(json.dumps({"id": x["id"], "metodo": metodo, "ruta": r["ruta"],
                                "salida_cruda": r["salida_cruda"]}, ensure_ascii=False, default=str) + "\n")
        print(f"lab {x['id']:>2} | {metodo:<18} | {r['estado']:<20} | {r['tiempo_s']} s")


lab  0 | baseline           | pisa_muro            | 26.3 s
lab  0 | agente             | exito                | 22.1 s
lab  0 | agente_sin_memoria | limite_pasos         | 163.5 s
lab  0 | aleatorio          | limite_pasos         | 0.0 s
lab  1 | baseline           | pisa_muro            | 42.1 s
lab  1 | agente             | propuestas_invalidas | 9.8 s
lab  1 | agente_sin_memoria | limite_pasos         | 162.7 s
lab  1 | aleatorio          | limite_pasos         | 0.0 s
lab  2 | baseline           | pisa_muro            | 39.5 s
lab  2 | agente             | propuestas_invalidas | 31.4 s
lab  2 | agente_sin_memoria | limite_pasos         | 109.3 s
lab  2 | aleatorio          | limite_pasos         | 0.0 s
lab  3 | baseline           | pisa_muro            | 22.8 s
lab  3 | agente             | limite_pasos         | 208.2 s
lab  3 | agente_sin_memoria | limite_pasos         | 107.4 s
lab  3 | aleatorio          | exito                | 0.0 s
lab  4 | baseline           | salto_no_a

## 9. Resumen de resultados (para la tabla del PDF)

In [ ]:
df = pd.read_csv(CSV)
print(f"N = {df['id'].nunique()} laberintos 9x9 | modelo {MODEL_NAME} @ {MODEL_REVISION}\n")
resumen = df.groupby("metodo").agg(
    tasa_exito=("exito", "mean"),
    tasa_exito_estricto=("exito_estricto", "mean"),
    pasos_prom=("pasos", "mean"),
    invalidas_prom=("invalidas", "mean"),
    valor_mal_prom=("valor_mal_declarado", "mean"),
    coord_mal_prom=("coordenada_mal_declarada", "mean"),
    tiempo_prom_s=("tiempo_s", "mean")).reindex(METODOS)
print(resumen.round(2), "\n")
print("Tipos de resultado por método:")
print(pd.crosstab(df["metodo"], df["estado"]).reindex(METODOS))
for m in ["agente", "agente_sin_memoria"]:
    ok = df[(df.metodo == m) & df.exito]
    if len(ok):
        print(f"\nEficiencia {m} en éxitos (pasos / óptimo): {(ok.pasos / ok.optimo).mean():.2f}")


N = 20 laberintos 9x9 | modelo Qwen/Qwen2.5-1.5B-Instruct @ 989aa7980e4cf806f80c7fef2b1adb7bc71aa306

                    tasa_exito  tasa_exito_estricto  pasos_prom  \
metodo                                                            
baseline                  0.00                 0.00        1.75   
agente                    0.10                 0.00       10.40   
agente_sin_memoria        0.00                 0.00       81.00   
aleatorio                 0.25                 0.25       75.15   

                    invalidas_prom  valor_mal_prom  coord_mal_prom  \
metodo                                                               
baseline                       NaN             NaN             NaN   
agente                       10.15            0.00            0.95   
agente_sin_memoria           34.00            0.05            4.20   
aleatorio                     0.00            0.00            0.00   

                    tiempo_prom_s  
metodo                             
ba

## 10. Caso de fallo
Lista los laberintos donde el agente (con memoria) falló y vuelve a ejecutar uno paso a paso. La decodificación es greedy, así que el resultado es el mismo de la evaluación. `analizar_fallo` entrega datos para explicar **por qué** falla: qué celdas revisita y en qué callejones sin salida entra.

In [ ]:
df = pd.read_csv(CSV)
fallos = df[(df.metodo == "agente") & ~df.exito]
print("Laberintos donde falla el agente:")
print(fallos[["id", "estado", "pasos", "optimo", "invalidas"]].to_string(index=False))

ID_FALLO = int(fallos.id.iloc[0]) if len(fallos) else None   # <- cambien el id si quieren mostrar otro
if ID_FALLO is not None:
    x = next(d for d in construir_dataset(N) if d["id"] == ID_FALLO)
    lab, a, b = x["laberinto"], tuple(x["inicio"]), tuple(x["meta"])
    print(f"\n===== Caso de fallo: laberinto {ID_FALLO} ({x['origen']}) | inicio {a} meta {b} | óptimo {len(bfs(lab,a,b))-1} pasos")
    dibujar(lab, [], a, b)
    r = correr_agente(generar, lab, a, b, verbose=True)
    print("\nVEREDICTO:", r["estado"], "| pasos:", r["pasos"], "| inválidas:", r["invalidas"])
    dibujar(lab, r["ruta"], a, b)
    analizar_fallo(lab, r["ruta"])


Laberintos donde falla el agente:
 id               estado  pasos  optimo  invalidas
  1 propuestas_invalidas      2      11        5.0
  2 propuestas_invalidas     11      10       12.0
  3         limite_pasos     81      13       73.0
  4 propuestas_invalidas      3      11        7.0
  5 propuestas_invalidas      1      12        5.0
  6 propuestas_invalidas      4      16        8.0
  8 propuestas_invalidas      2      11        6.0
  9 propuestas_invalidas      6      21        5.0
 10 propuestas_invalidas      4      13        7.0
 11 propuestas_invalidas      3      17        7.0
 12 propuestas_invalidas      1      16        4.0
 13 propuestas_invalidas     10      14        6.0
 14 propuestas_invalidas      5      16        4.0
 15 propuestas_invalidas     10      12       12.0
 16 propuestas_invalidas     16      22        9.0
 17 propuestas_invalidas      8      11        7.0
 18 propuestas_invalidas      5      12        7.0
 19 propuestas_invalidas     14      11       12

## 11. Demo para el video
Laberinto con semilla aleatoria (se imprime, para que sea verificable y no elegido a dedo). Muestra baseline y agente sobre el mismo input.

In [ ]:
import random as _r
seed_demo = _r.randint(10_000, 99_999)
lab, a, b = generar_laberinto(seed_demo)
print(f"Semilla de la demo: {seed_demo} | inicio {a} | meta {b} | óptimo {len(bfs(lab,a,b))-1} pasos\n")
dibujar(lab, [], a, b)

print("\n========== BASELINE (direct prompting) ==========")
rb = correr_baseline(generar, lab, a, b)
print(rb["salida_cruda"][:1200])
print("\nVEREDICTO BASELINE:", rb["estado"])
dibujar(lab, rb["ruta"], a, b)

print("\n========== AGENTE PASO A PASO ==========")
ra = correr_agente(generar, lab, a, b, verbose=True)
print("\nVEREDICTO AGENTE:", ra["estado"], "| pasos:", ra["pasos"], "| inválidas:", ra["invalidas"],
      "| éxito estricto:", ra["estado"] == "exito" and ra["invalidas"] == 0)
dibujar(lab, ra["ruta"], a, b)

Semilla de la demo: 55409 | inicio (6, 1) | meta (2, 8) | óptimo 15 pasos

   █              
   █ █ █   █ █ █  
               █ M
 █ █   █ █ █   █  
               █  
   █ █ █   █ █ █  
   I   █          
 █ █   █   █ █ █  
           █      

========== BASELINE (direct prompting) ==========
### Paso 1: Inicio del Laberinto
**Coordenada:** (6, 1)
**Valor Real:** 1
**Adyacencia:** No hay adyacencias válidas desde esta posición ya que no existe ninguna fila o columna directamente superior o inferior.

### Paso 2: Movimiento hacia arriba
**Coordenada:** (5, 1)
**Valor Real:** 0
**Adyacencia:** Acierta, ahora se encuentra en la fila 5.

### Paso 3: Movimiento hacia derecha
**Coordenada:** (5, 2)
**Valor Real:** 0
**Adyacencia:** Acierta, ahora se encuentra en la columna 2.

### Paso 4: Movimiento hacia abajo
**Coordenada:** (6, 2)
**Valor Real:** 0
**Adyacencia:** Acierta, ahora se encuentra en la fila 6.

### Paso 5: Movimiento hacia izquierda
**Coordenada:** (5, 1)
**Valor Real:** 0
